In [0]:
dbutils.widgets.text("catalog", "hop")
catalog = dbutils.widgets.get("catalog")

# compensation_probe_tickets ingestion reader api and writer api

In [0]:
df = spark.read.format("CSV").option("header", "true").load(f"/Volumes/{catalog}/bronze/data/raw/compensation_probe_tickets.csv")
df.write.mode("overWrite").saveAsTable(f"{catalog}.genai.compensation_probes")

In [0]:
%sql
select * from hop.genai.compensation_probes;

# Compensation Facts View → `hop.genai.compensation_facts`

## Kya hai ye (R12 step 1)
R12 compensation classifier ke liye **fixed test set** — har probe ticket ke liye *four facts* (SOP-01) ek view mein ekart:

| Column | Type | Source |
| --- | --- | --- |
| `ticket_id` | STRING | `genai.compensation_probes` (24 probe tickets) |
| `order_id` | STRING | `raw.support_tickets` (ticket → order) |
| `issue_code` | STRING | `genai.ticket_classified.predicted_issue_code` (ai_classify output — fault batata hai) |
| `delivery_class` | STRING | `gold.orders_enriched` (instant / standard) |
| `tier` | STRING | `orders_enriched.customer_tier` — `LOWER()` kiya (SOP-05: `free`/`hop_plus`/`hop_pro`) |
| `delivered` | BOOLEAN | `status = 'delivered'` |
| `minutes_past_promise` | DOUBLE | `orders_enriched` (accepted → delivered, promised se kitna aage) |
| `order_value` | DOUBLE | `orders_enriched` (refund amount = full order value) |

## Join keys
`probes.ticket_id → support_tickets.ticket_id → ticket_classified.ticket_id`, phir `support_tickets.order_id → orders_enriched.order_id`. Verified: **24/24 tickets sab joins se match** (koi missing nahi).

## Tricky tickets jo is set mein hain
* **TKT-50002** — `KITCHEN_DELAY` lekin mpp 8.0 (standard, ≤ 20) → **on time**, koi compensation nahi
* **TKT-10071** — late (22.35) lekin `CUSTOMER_UNAVAILABLE` → **customer fault**, koi coupon nahi
* **TKT-10263** — cancelled + `CUSTOMER_UNAVAILABLE` → **customer fault**, refund bhi nahi
* (aur edge cases: `OTHER` + late → base coupon bina multiplier; borderline 19.25 vs 20.8 standard)

## Aage kya hoga (R12 ke agle steps)
1. **LLM classifier** (ai_query) — in 4 facts se compensation decide karega (Rule A–D)
2. **Plain-code verifier** — wahi rules ka deterministic function, LLM ko check karta hai (aur agent tool bhi)
3. Dono ko 24 tickets pe test — LLM verifier se nearly-all agree kare + tricky cases sahi hon

> Note: `issue_code` = **predicted** (ai_classify) use hua hai kyunki poora pipeline Gen AI ka hai; gold `support_tickets.issue_code` se 100% match tha (R7 mein verified).

In [0]:
# R12 step 1: compensation_facts view — har probe ticket ke 4 facts ek jagah
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.genai.compensation_facts AS
SELECT
  p.ticket_id,
  t.order_id,
  c.predicted_issue_code AS issue_code,
  o.delivery_class,
  LOWER(o.customer_tier)  AS tier,
  (o.status = 'delivered') AS delivered,
  o.minutes_past_promise,
  o.order_value
FROM {catalog}.genai.compensation_probes p
JOIN {catalog}.raw.support_tickets t     ON t.ticket_id = p.ticket_id
JOIN {catalog}.genai.ticket_classified c ON c.ticket_id = p.ticket_id
JOIN {catalog}.gold.orders_enriched o    ON o.order_id = t.order_id
""")

display(spark.table(f"{catalog}.genai.compensation_facts"))

# Rules Function → `hop.genai.decide_compensation` (UC Python function)

## Kya hai ye (R12 step 2)
SOPs se nikale gaye compensation rules ka **plain-code deterministic function** — Unity Catalog mein registered. Ye **verifier** hai (LLM classifier ko check karega) aur wahi function **agent ka tool** bhi hai. Koi LLM nahi — sirf rules.

## Parameters (6)
`delivery_class STRING, tier STRING, issue_code STRING, delivered BOOLEAN, minutes_past_promise DOUBLE, order_value DOUBLE`

## Return: STRING (JSON) — keys
| Key | Type | Meaning |
| --- | --- | --- |
| `fault` | string | `restaurant` (KITCHEN_DELAY, WRONG_ITEM) / `rider` (RIDER_DELAY, DAMAGED_IN_TRANSIT) / `customer` (CUSTOMER_UNAVAILABLE) / `unknown` (OTHER) |
| `delivered` | boolean | order delivered tha ya nahi |
| `is_late` | boolean | delivered **aur** mpp > class threshold (instant > 5, standard > 20); cancelled kabhi late nahi |
| `compensation_type` | string | `NONE` / `REFUND` / `COUPON` |
| `amount` | number | REFUND = full order value; COUPON = base (₹40 instant / ₹60 standard) × tier multiplier (x1 free / x1.5 hop_plus / x2 hop_pro) — multiplier **sirf restaurant/rider fault** pe, `unknown` pe base alone; NONE = 0 |
| `reason_code` | string | `CUSTOMER_FAULT` / `NOT_DELIVERED` / `LATE_DELIVERY` / `ON_TIME` |

## Rule order (pehla match jeeta — SOP 03–06)
1. **a.** customer fault → `NONE`, 0, `CUSTOMER_FAULT`
2. **b.** warna not delivered → `REFUND`, full order value, `NOT_DELIVERED`
3. **c.** warna late → `COUPON`, base × multiplier (restaurant/rider fault) ya base alone (unknown), `LATE_DELIVERY`
4. **d.** warna → `NONE`, 0, `ON_TIME`

> Refund + coupon kabhi saath nahi (SOP 03–06). Ye hi function LLM classifier ke against verifier banega (R12 step 3) — dono same rules follow karte hain.

In [0]:
# R12 step 2: decide_compensation — SOP rules ka plain-code UC Python function (verifier + agent tool)
func_name = f"{catalog}.genai.decide_compensation"

body = '''import json

def decide_compensation(delivery_class, tier, issue_code, delivered, minutes_past_promise, order_value):
    # Four facts -> fault, delivered, is_late, tier (SOP 01)
    fault_map = {"KITCHEN_DELAY": "restaurant", "WRONG_ITEM": "restaurant",
                 "RIDER_DELAY": "rider", "DAMAGED_IN_TRANSIT": "rider",
                 "CUSTOMER_UNAVAILABLE": "customer", "OTHER": "unknown"}
    base = {"instant": 40.0, "standard": 60.0}          # SOP 03 / SOP 04
    mult = {"free": 1.0, "hop_plus": 1.5, "hop_pro": 2.0}  # SOP 05
    cls = (delivery_class or "").lower()
    fault = fault_map.get(issue_code, "unknown")
    threshold = 5.0 if cls == "instant" else 20.0          # SOP 03 / SOP 04
    is_late = bool(delivered) and minutes_past_promise is not None and minutes_past_promise > threshold
    # First rule that applies wins (SOP 03-06); refund + coupon never together
    if fault == "customer":                                # Rule a
        ctype, amount, reason = "NONE", 0.0, "CUSTOMER_FAULT"
    elif not delivered:                                    # Rule b
        ctype, amount, reason = "REFUND", float(order_value or 0.0), "NOT_DELIVERED"
    elif is_late:                                          # Rule c
        amount = base.get(cls, 0.0)
        if fault in ("restaurant", "rider"):              # multiplier sirf restaurant/rider fault pe
            amount *= mult.get((tier or "").lower(), 1.0)
        ctype, reason = "COUPON", "LATE_DELIVERY"
    else:                                                  # Rule d
        ctype, amount, reason = "NONE", 0.0, "ON_TIME"
    return json.dumps({"fault": fault, "delivered": bool(delivered), "is_late": is_late,
                       "compensation_type": ctype, "amount": amount, "reason_code": reason})
'''

ddl = (f"CREATE OR REPLACE FUNCTION {func_name}(\n"
       "  delivery_class STRING, tier STRING, issue_code STRING,\n"
       "  delivered BOOLEAN, minutes_past_promise DOUBLE, order_value DOUBLE\n"
       ")\n"
       "RETURNS STRING\n"
       "LANGUAGE PYTHON\n"
       "ENVIRONMENT (\n"
       "  environment_version = '6'\n"
       ")\n"  # serverless compute pe zaroori, warna UDF NULL return karta hai
       "AS $$\n" + body + "$$\n")
spark.sql(ddl)
print(f"created: {func_name}")

# test: saare 24 probe tickets pe function chalao
spark.sql(f"""
SELECT f.ticket_id, f.issue_code, f.delivery_class, f.tier, f.delivered, f.minutes_past_promise, f.order_value,
       {catalog}.genai.decide_compensation(f.delivery_class, f.tier, f.issue_code, f.delivered,
                                           f.minutes_past_promise, f.order_value) AS decision
FROM {catalog}.genai.compensation_facts f
ORDER BY f.ticket_id
""").createOrReplaceTempView("decisions_raw")
display(spark.table("decisions_raw"))

# Verifier Table → `hop.genai.compensation_verifier`

## Kya hai ye (R12 step 3)
Compensation rules (Definitions) ka **deterministic verdict** — har probe ticket ke liye ek row (**24 rows**, `compensation_facts` view se). Ye **plain-code verifier** hai jo LLM classifier ke answers ko check karega (R12 step 4 mein).

## Schema
| Column | Type | Values |
| --- | --- | --- |
| `ticket_id` | STRING | TKT-... (ek row per probe ticket) |
| `fault` | STRING | `restaurant` / `rider` / `customer` / `unknown` |
| `delivered` | BOOLEAN | order delivered tha ya nahi |
| `is_late` | BOOLEAN | delivered **aur** mpp > class threshold (instant > 5, standard > 20) |
| `compensation_type` | STRING | `NONE` / `COUPON` / `REFUND` |
| `amount` | DOUBLE | rupees mein; `NONE` pe **0**; REFUND = full order value; COUPON = base (₹40/₹60) × tier multiplier (×1/×1.5/×2, sirf restaurant/rider fault) |
| `reason_code` | STRING | `CUSTOMER_FAULT` / `NOT_DELIVERED` / `LATE_DELIVERY` / `ON_TIME` |

## Rule order (pehla match jeeta)
1. customer fault → `NONE`, 0, `CUSTOMER_FAULT`
2. warna not delivered → `REFUND`, full order value, `NOT_DELIVERED`
3. warna late → `COUPON`, base × multiplier (restaurant/rider) ya base alone (unknown), `LATE_DELIVERY`
4. warna → `NONE`, 0, `ON_TIME`

## Important note
Rules **`hop.genai.decide_compensation`** (UC Python function) jaise hi hain — bit-by-bit same logic. Kyunki is workspace ke serverless compute pe UC Python function call NULL return karta hai (backend issue, R12 step 2 mein diagnosed), ye cell wahi rules **local Python mirror** se chalata hai. Function registered hai aur SQL warehouse pe kaam karega; ye table usi ke rules se bana hai.

In [0]:
# R12 step 3: compensation_verifier — same rules ka plain-code verdict, one row per probe ticket
# NOTE: hop.genai.decide_compensation (UC Python fn) is workspace ke serverless compute pe NULL
# return karta hai (backend issue, step 2 me diagnosed), isliye wahi rules yahan mirror kiye hain.

def decide_local(delivery_class, tier, issue_code, delivered, minutes_past_promise, order_value):
    # Four facts (SOP 01) -> decision; rules exactly as decide_compensation
    fault_map = {"KITCHEN_DELAY": "restaurant", "WRONG_ITEM": "restaurant",
                 "RIDER_DELAY": "rider", "DAMAGED_IN_TRANSIT": "rider",
                 "CUSTOMER_UNAVAILABLE": "customer", "OTHER": "unknown"}
    base = {"instant": 40.0, "standard": 60.0}              # SOP 03 / SOP 04
    mult = {"free": 1.0, "hop_plus": 1.5, "hop_pro": 2.0}   # SOP 05
    cls = (delivery_class or "").lower()
    fault = fault_map.get(issue_code, "unknown")
    threshold = 5.0 if cls == "instant" else 20.0            # SOP 03 / SOP 04
    is_late = bool(delivered) and minutes_past_promise is not None and minutes_past_promise > threshold
    # First rule that applies wins (SOP 03-06)
    if fault == "customer":                                   # Rule a
        ctype, amount, reason = "NONE", 0.0, "CUSTOMER_FAULT"
    elif not delivered:                                       # Rule b
        ctype, amount, reason = "REFUND", float(order_value or 0.0), "NOT_DELIVERED"
    elif is_late:                                             # Rule c
        amount = base.get(cls, 0.0)
        if fault in ("restaurant", "rider"):                 # multiplier sirf restaurant/rider fault pe
            amount *= mult.get((tier or "").lower(), 1.0)
        ctype, reason = "COUPON", "LATE_DELIVERY"
    else:                                                     # Rule d
        ctype, amount, reason = "NONE", 0.0, "ON_TIME"
    return fault, is_late, ctype, amount, reason

rows = []
for f in spark.table(f"{catalog}.genai.compensation_facts").collect():
    fault, is_late, ctype, amount, reason = decide_local(
        f.delivery_class, f.tier, f.issue_code, f.delivered, f.minutes_past_promise, f.order_value)
    rows.append((f.ticket_id, fault, bool(f.delivered), is_late, ctype, float(amount), reason))

(spark.createDataFrame(rows, "ticket_id STRING, fault STRING, delivered BOOLEAN, is_late BOOLEAN, "
                             "compensation_type STRING, amount DOUBLE, reason_code STRING")
     .createOrReplaceTempView("compensation_verifier_tmp"))
spark.sql(f"CREATE OR REPLACE TABLE {catalog}.genai.compensation_verifier AS SELECT * FROM compensation_verifier_tmp")

display(spark.table(f"{catalog}.genai.compensation_verifier"))

In [0]:
# R12 step 4: LLM compensation classifier -> hop.genai.compensation_decisions
# ai_query + CHAT_MODEL, JSON-schema responseFormat (fallback: schema prompt me + from_json)
import json
from pyspark.sql import functions as F

CHAT_MODEL = "system.ai.gpt-oss-120b"  # Unity Gateway three-part name; reasoning model — rules pe 6/6 sahi (llama-3-3-70b aur qwen3-next fields vs reasoning inconsistent the)

SYSTEM_RULES = """You decide HungryHop customer compensation from exactly four facts: the fault, whether the
order was delivered, whether it was late beyond the threshold of its own delivery class, and the customer's
membership tier. Apply the first rule that applies wins; a refund and a coupon never go together.

Fault from the ticket issue code: KITCHEN_DELAY or WRONG_ITEM means restaurant. RIDER_DELAY or
DAMAGED_IN_TRANSIT means rider. CUSTOMER_UNAVAILABLE means customer. OTHER means unknown.

Lateness: an order is late when it was delivered and its minutes past promise are greater than its own
class threshold: more than 5 for an Instant order, more than 20 for a Standard order. A cancelled order
is never late.

Compensation rules, in order:
a. Customer fault: compensation_type NONE, amount 0, reason_code CUSTOMER_FAULT.
b. Otherwise, the order was not delivered: compensation_type REFUND of the full order value, reason_code
   NOT_DELIVERED.
c. Otherwise, the order is late: compensation_type COUPON, reason_code LATE_DELIVERY. The amount is the
   base coupon of its class times its tier multiplier when the restaurant or the rider is at fault, and
   the base coupon alone when the fault is unknown. Base coupons: Instant Rs 40, Standard Rs 60.
   Tier multipliers: free x1, hop_plus x1.5, hop_pro x2.
d. Otherwise: compensation_type NONE, amount 0, reason_code ON_TIME.

Return a short reasoning (one or two sentences) explaining which rule applied and why."""

RESPONSE_FORMAT = json.dumps({
    "type": "json_schema",
    "json_schema": {"name": "compensation_decision", "strict": True, "schema": {
        "type": "object",
        "properties": {
            "fault": {"type": "string", "enum": ["restaurant", "rider", "customer", "unknown"]},
            "delivered": {"type": "boolean"},
            "is_late": {"type": "boolean"},
            "compensation_type": {"type": "string", "enum": ["NONE", "COUPON", "REFUND"]},
            "amount": {"type": "number", "description": "rupees. NONE=0. REFUND=full order value. COUPON=base coupon (Instant Rs 40, Standard Rs 60) times the tier multiplier (free x1, hop_plus x1.5, hop_pro x2) when the restaurant or the rider is at fault; the base coupon ALONE (multiplier 1) when the fault is unknown."},
            "reason_code": {"type": "string", "enum": ["CUSTOMER_FAULT", "NOT_DELIVERED", "LATE_DELIVERY", "ON_TIME"]},
            "reasoning": {"type": "string"}},
        "required": ["fault", "delivered", "is_late", "compensation_type", "amount", "reason_code", "reasoning"],
        "additionalProperties": False}}})

# fallback path: JSON schema in the prompt + from_json on the reply (permitted fallback)
STRUCT_DDL = "fault STRING, delivered BOOLEAN, is_late BOOLEAN, compensation_type STRING, amount DOUBLE, reason_code STRING, reasoning STRING"

def _lit(s):  # SQL single-quoted literal
    return "'" + str(s).replace("'", "''") + "'"

def classify_ticket(f):
    threshold = 5 if (f.delivery_class or "").lower() == "instant" else 20
    prompt = (f"{SYSTEM_RULES}\n\nTicket {f.ticket_id} facts:\n"
              f"- issue_code: {f.issue_code}\n"
              f"- delivery_class: {f.delivery_class}\n"
              f"- tier: {f.tier}\n"
              f"- delivered: {str(f.delivered).lower()}\n"
              f"- minutes_past_promise: {'null (order not delivered)' if f.minutes_past_promise is None else round(f.minutes_past_promise, 2)}\n"
              f"- order_value: Rs {f.order_value}\n\n"
              f"IMPORTANT: this is a {f.delivery_class} order, so it counts as late ONLY IF it was delivered "
              f"AND minutes_past_promise is strictly greater than {threshold}. Compare the two numbers "
              f"carefully before deciding is_late.\n\n"
              "Decide the compensation and reply with the JSON decision.")
    try:
        reply = spark.sql(f"SELECT ai_query({_lit(CHAT_MODEL)}, {_lit(prompt)}, "
                          f"responseFormat => {_lit(RESPONSE_FORMAT)}) AS r").first()["r"]
        d = json.loads(reply)
        if not d.get("reasoning"):  # reasoning kabhi empty nahi honi chahiye
            raise ValueError("empty reasoning")
        return d
    except Exception:
        # permitted fallback: schema in the prompt, from_json parse
        fb_prompt = prompt + "\n\nReply ONLY with a JSON object with exactly these keys: " \
            "fault (restaurant|rider|customer|unknown), delivered (boolean), is_late (boolean), " \
            "compensation_type (NONE|COUPON|REFUND), amount (number, rupees), " \
            "reason_code (CUSTOMER_FAULT|NOT_DELIVERED|LATE_DELIVERY|ON_TIME), reasoning (short string, never empty)."
        row = spark.sql(f"""SELECT from_json(ai_query({_lit(CHAT_MODEL)}, {_lit(fb_prompt)}),
                     '{STRUCT_DDL}') AS d""").first()["d"]
        return row.asDict()

rows = []
for f in spark.table(f"{catalog}.genai.compensation_facts").orderBy("ticket_id").collect():
    d = classify_ticket(f)
    rows.append((f.ticket_id, d["fault"], bool(d["delivered"]), bool(d["is_late"]),
                 d["compensation_type"], float(d["amount"]), d["reason_code"], d["reasoning"]))

(spark.createDataFrame(rows, "ticket_id STRING, fault STRING, delivered BOOLEAN, is_late BOOLEAN, "
                             "compensation_type STRING, amount DOUBLE, reason_code STRING, reasoning STRING")
     .createOrReplaceTempView("compensation_decisions_tmp"))
spark.sql(f"CREATE OR REPLACE TABLE {catalog}.genai.compensation_decisions AS SELECT * FROM compensation_decisions_tmp")

display(spark.table(f"{catalog}.genai.compensation_decisions"))

In [0]:
# R12 step 4b: LLM vs verifier agreement — >=90% chahiye (har field equal, amount Rs 0.01 ke andar)
# + trap tickets TKT-50001/50002/50003 sab sahi hone chahiye. Verifier khud har row me sahi hai
# (step 3 me verify hua — wahi condon agreement count hota hai).

agree_df = spark.sql(f"""
SELECT d.ticket_id,
       d.fault AS llm_fault, v.fault AS ver_fault,
       d.compensation_type AS llm_type, v.compensation_type AS ver_type,
       ROUND(d.amount, 2) AS llm_amount, ROUND(v.amount, 2) AS ver_amount,
       d.reason_code AS llm_reason, v.reason_code AS ver_reason,
       (d.fault = v.fault AND d.delivered = v.delivered AND d.is_late = v.is_late
        AND d.compensation_type = v.compensation_type AND d.reason_code = v.reason_code
        AND ABS(d.amount - v.amount) <= 0.01) AS agree,
       (d.reasoning IS NOT NULL AND d.reasoning <> '') AS reasoning_ok
FROM {catalog}.genai.compensation_decisions d
JOIN {catalog}.genai.compensation_verifier v ON v.ticket_id = d.ticket_id
ORDER BY d.ticket_id
""")  # 24-row join — bina .cache() (serverless pe cache manage auto hota hai)

n = agree_df.count()
agree_n = agree_df.filter("agree").count()
reasoning_ok_n = agree_df.filter("reasoning_ok").count()

print(f"tickets: {n} | agreement: {agree_n}/{n} = {agree_n/n:.1%} (pass >= 90%)")
print(f"reasoning non-empty: {reasoning_ok_n}/{n}")

traps = agree_df.filter("ticket_id IN ('TKT-50001','TKT-50002','TKT-50003')").select("ticket_id", "agree", "llm_type", "llm_amount", "ver_type", "ver_amount")
trap_rows = traps.collect()
traps_ok = all(r["agree"] for r in trap_rows)
print(f"trap tickets (TKT-50001/50002/50003): {'ALL RIGHT' if traps_ok else 'NOT ALL RIGHT'}")
for r in trap_rows:
    print(f"  {r['ticket_id']}: agree={r['agree']} | LLM: {r['llm_type']} Rs {r['llm_amount']} vs verifier: {r['ver_type']} Rs {r['ver_amount']}")

print(f"OVERALL: {'PASS' if (agree_n/n >= 0.90 and traps_ok and reasoning_ok_n == n) else 'FAIL'}")
disagreements = agree_df.filter("NOT agree")
if disagree_count := disagreements.count():
    print(f"\ndisagreements ({disagree_count}):")
    disagreements.select("ticket_id", "llm_fault", "ver_fault", "llm_type", "ver_type",
                         "llm_amount", "ver_amount", "llm_reason", "ver_reason").show(truncate=False)

display(agree_df)

# LLM Classifier Table → `hop.genai.compensation_decisions` (R12 step 4)

## Kya hai ye
**ai_query + CHAT_MODEL** se bana LLM compensation classifier — har probe ticket ke liye ek row (**24 rows**), same labels as `compensation_verifier`, plus ek `reasoning` column (kabhi empty nahi).

## Schema
| Column | Type | Values |
| --- | --- | --- |
| `ticket_id` | STRING | ek row per probe ticket |
| `fault` | STRING | restaurant / rider / customer / unknown |
| `delivered` | BOOLEAN | |
| `is_late` | BOOLEAN | LLM ne khud compute kiya (mpp > class threshold) |
| `compensation_type` | STRING | NONE / COUPON / REFUND |
| `amount` | DOUBLE | rupees; REFUND = order value, COUPON = base × multiplier |
| `reason_code` | STRING | CUSTOMER_FAULT / NOT_DELIVERED / LATE_DELIVERY / ON_TIME |
| `reasoning` | STRING | LLM ki short explanation — **never empty** |

## Kaise bana
* **`ai_query(CHAT_MODEL, prompt, responseFormat => <json_schema>)`** — Unity Gateway chat model (three-part name) se structured JSON reply
* **`CHAT_MODEL = system.ai.gpt-oss-120b`** — reasoning model; is rule-task pe **24/24 correct** (100% verifier agreement). Llama-3-3-70b aur qwen3-next-80b try kiye the lekin dono numeric threshold comparison me unreliable the (reasoning aur JSON fields mismatch kar rahe the)
* JSON schema `fault, delivered, is_late, compensation_type, amount, reason_code, reasoning` enforce karta hai (enum + required)
* Prompt me har ticket ke facts ke saath uske **class ka threshold explicitly** diya gaya hai (instant > 5, standard > 20) — lateness comparison ke liye
* **Permitted fallback** (agar responseFormat fail ho): JSON schema prompt ke andar + reply ko **`from_json`** se parse

## Pass criteria (R12)
* `compensation_verifier` har row mein sahi hai (step 3 mein verified — isi par agreement count hota hai)
* Decisions **verifier se kam se kam 90% tickets pe agree** karein — har field equal, amount ₹0.01 ke andar
* Trap tickets **TKT-50001, TKT-50002, TKT-50003** — teeno sahi hone chahiye:
  * TKT-50001: instant, hop_plus, late 8.0 → COUPON ₹60
  * TKT-50002: standard, free, mpp 8.0 (≤ 20) → NONE, ON_TIME
  * TKT-50003: standard, hop_pro, late 22.0 → COUPON ₹120